# LocalAudioBook — Kokoro render on GPU

Renders the chunk audio for one book and zips it for import back into LocalAudioBook.

**This notebook does no text processing.** `chunks.jsonl` already holds finished strings —
`preprocessText`, `splitIntoChunks`, `normaliseForSpeech`, the TTS warm-up prefix and the
512-token split all ran on your machine. Speak what you are given, nothing else: the chunk
count has to match `run.json` or the result will not be resumable.

### Before you run

1. Settings → **Accelerator: GPU T4 x2**, **Internet: On** (both need phone verification).
2. Upload `backend/audio/gpu-job/` (`job.json` + `chunks.jsonl`) as a dataset —
   **set it to Private.** It contains the book.
3. Point `JOB_DIR` below at it.
4. Use **Save & Run All**, not an interactive session: it survives a dropped connection,
   and interactive sessions time out on idle.

Re-running is safe — finished chunks are skipped, so a killed session resumes.

In [ ]:
# espeak-ng is not optional: kokoro imports fine without it and then produces nothing useful.
!apt-get -qq install -y espeak-ng > /dev/null
!pip -q install kokoro soundfile

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import json, os, glob

# The folder holding job.json and chunks.jsonl.
JOB_DIR = glob.glob('/kaggle/input/*')[0]
OUT_DIR = '/kaggle/working/chunks'
os.makedirs(OUT_DIR, exist_ok=True)

job = json.load(open(os.path.join(JOB_DIR, 'job.json')))
with open(os.path.join(JOB_DIR, 'chunks.jsonl')) as fh:
    chunks = [json.loads(line) for line in fh if line.strip()]

assert len(chunks) == job['total'], f"chunks.jsonl has {len(chunks)}, job.json says {job['total']}"

print(f"voice {job['voice']}  lang {job['langCode']}  speed {job['speed']}x  {job['sampleRate']}Hz")
print(f"{job['total']} chunks, {sum(len(c['pieces']) for c in chunks)} pieces, ~{job['wordCount']:,} words")
print(f"title: {job.get('title')}")

In [ ]:
import numpy as np, soundfile as sf, time
from kokoro import KPipeline

SR   = job['sampleRate']
JOIN = np.zeros(int(SR * job['joinSilenceMs'] / 1000), dtype=np.float32)
FLOOR = 10 ** (job['silenceFloorDbfs'] / 20)
LEAD  = int(SR * job['leadInMs'] / 1000)

try:
    pipe = KPipeline(lang_code=job['langCode'], device='cuda')
except TypeError:
    pipe = KPipeline(lang_code=job['langCode'])


def as_numpy(audio):
    if torch.is_tensor(audio):
        return audio.detach().cpu().numpy().astype(np.float32)
    return np.asarray(audio, dtype=np.float32)


def trim_silent_ends(wave):
    """Mirror of trimSilentEnds in engines/kokoro.js.

    Kokoro pads every utterance with ~303ms lead and ~497ms tail. Concatenated raw
    that leaves ~990ms between pieces - ten times a real chunk boundary, and an
    audible stall mid-paragraph.
    """
    loud = np.flatnonzero(np.abs(wave) > FLOOR)
    if loud.size == 0:
        return wave
    start = max(0, loud[0] - LEAD)
    end   = min(len(wave), loud[-1] + LEAD + 1)
    return wave[start:end]


def render(pieces):
    waves = []
    for piece in pieces:
        # One piece is one utterance locally. KPipeline may still split it internally,
        # so join those sub-segments tightly - the 400ms gap belongs between pieces.
        segments = [as_numpy(audio) for _, _, audio in pipe(piece, voice=job['voice'], speed=job['speed'])]
        if segments:
            waves.append(np.concatenate(segments) if len(segments) > 1 else segments[0])

    if not waves:
        raise RuntimeError('no audio produced')
    if len(waves) == 1:
        return waves[0]

    parts = []
    for k, wave in enumerate(waves):
        parts.append(trim_silent_ends(wave))
        if k < len(waves) - 1:
            parts.append(JOIN)
    return np.concatenate(parts)


print('ready')

In [ ]:
started = time.time()
audio_sec = 0.0
done = 0
skipped = 0

for rec in chunks:
    out = os.path.join(OUT_DIR, f"chunk-{rec['i'] + 1:04d}.flac")
    if os.path.exists(out):
        skipped += 1
        continue

    wave = render(rec['pieces'])

    # Clamp here, at the source. Kokoro's vocoder can emit a sample thousands of times
    # full scale; downstream that poisons both the peak and the RMS measurement and the
    # whole chunk gets levelled into silence. int16 FLAC cannot carry it out of this cell.
    wave = np.clip(wave, -1.0, 1.0)

    tmp = out + '.part'
    sf.write(tmp, wave, SR, format='FLAC', subtype='PCM_16')
    os.replace(tmp, out)

    audio_sec += len(wave) / SR
    done += 1

    if done % 10 == 0 or done == 1:
        spent = time.time() - started
        rate  = audio_sec / spent if spent else 0
        left  = (len(chunks) - skipped - done) * (spent / done)
        print(f"{done + skipped}/{len(chunks)}  {rate:.1f}x realtime  "
              f"{audio_sec / 60:.0f}min audio  ~{left / 60:.0f}min left", flush=True)

spent = time.time() - started
print(f"\ndone: {done} rendered, {skipped} already present")
if done:
    print(f"{audio_sec / 3600:.2f}h audio in {spent / 60:.1f}min  =  {audio_sec / spent:.1f}x realtime")

### Reading the multiplier

- **30x+** — working as intended.
- **10-30x** — fine, but batching pieces would help if you do this often.
- **under 10x** — the GPU is not the bottleneck. It is almost always espeak phonemisation
  on Kaggle's weak vCPUs. Check the accelerator is actually on before anything else.

In [ ]:
# Zip for download. Do NOT write chunk-NNNN.json sidecars alongside these files -
# the local pass treats a sidecar as proof the chunk is already conditioned and will
# skip levelling, fades, gap padding and the word timeline for it, permanently.
!cd /kaggle/working && rm -f chunks.zip && zip -qr chunks.zip chunks && ls -lh chunks.zip
print(len(glob.glob(f'{OUT_DIR}/*.flac')), 'chunks of', job['total'])

### Back on your machine

```
unzip chunks.zip
npm run gpu:import -- <the extracted chunks folder>
```

Then press **Resume** in the sidebar. Conditioning, the word timeline and the merge all
run locally; no synthesis will happen.

Don't edit the book between exporting and importing — `runKey` hashes the preprocessed
text, so an edit changes the run's identity and Resume will refuse with
`CHUNKS_FROM_ANOTHER_RUN`.